## 19_core_scaling_decomposition
  Decomposition beta_Y = beta_E + beta_P on 115 regions (2023), and the commuting block on CZ + PL
  (2021): employment identity E = R + I - X, Delta_comm, gravity/radiation null models with
  cross-fitting, excess attraction regression.
  Output: results/core_scaling_decomposition/*.csv. Environment: nuts3-mobility.

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

# ---- Config (change names and paths here only) ----
ROOT_MARKER = Path("src") / "config.py"      # file that exists only in the project root
MASTER_PATH = Path("data/processed/nuts3_master_table.parquet")
OUT_DIR = Path("results/core_scaling_decomposition")
YEAR = 2023
COL_ID, COL_YEAR, COL_COUNTRY = "nuts3_id", "year", "country_code"
COL_N, COL_E, COL_Y = "population", "employment", "gdp_total"
COV_TYPE = "HC3"
# ----------------------------------------------------

# Project root: walk up from the working folder until ROOT_MARKER is found
start = Path.cwd()
candidates = [start, *start.parents]
root = next((p for p in candidates if (p / ROOT_MARKER).is_file()), None)
if root is None:
    raise FileNotFoundError(
        f"Nem található a {ROOT_MARKER.as_posix()} fájl a munkakönyvtár egyik "
        "szülőmappájában sem - indítsd a notebookot a projekten belülről."
    )
os.chdir(root)
print(f"Munkakönyvtár: a projekt gyökere ({candidates.index(root)} szinttel feljebb).")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Read only the needed columns (skips the geometry column)
cols = [COL_ID, COL_YEAR, COL_COUNTRY, COL_N, COL_E, COL_Y]
df = pd.read_parquet(MASTER_PATH, columns=cols)
df = df[df[COL_YEAR] == YEAR].copy()
n_year = len(df)

# One common sample for all three regressions: all three variables positive
mask = (df[[COL_N, COL_E, COL_Y]].notna().all(axis=1)
        & (df[[COL_N, COL_E, COL_Y]] > 0).all(axis=1))
df = df[mask].copy()
print(f"Regions in {YEAR}: {n_year}, in common sample: {len(df)}")
print(df[COL_COUNTRY].value_counts().to_string())

# Logs; productivity P = Y / E, so ln P = ln Y - ln E
df["ln_n"] = np.log(df[COL_N])
df["ln_y"] = np.log(df[COL_Y])
df["ln_e"] = np.log(df[COL_E])
df["ln_p"] = df["ln_y"] - df["ln_e"]

# Same regressors (ln N + country fixed effects) in all three models
rows = []
for label, dep in [("beta_Y", "ln_y"), ("beta_E", "ln_e"), ("beta_P", "ln_p")]:
    res = smf.ols(f"{dep} ~ ln_n + C({COL_COUNTRY})", data=df).fit(cov_type=COV_TYPE)
    lo, hi = res.conf_int().loc["ln_n"]
    rows.append({"quantity": label, "slope": res.params["ln_n"],
                 "se_" + COV_TYPE: res.bse["ln_n"],
                 "ci95_low": lo, "ci95_high": hi, "n": int(res.nobs)})

out = pd.DataFrame(rows)
print(out.round(4).to_string(index=False))

# Identity check: beta_Y - beta_E - beta_P should be ~0 (floating-point level)
s = out.set_index("quantity")["slope"]
print(f"beta_Y - (beta_E + beta_P) = {s['beta_Y'] - s['beta_E'] - s['beta_P']:.2e}")

out.to_csv(OUT_DIR / "core_scaling_decomposition.csv", index=False)
print("Saved:", (OUT_DIR / "core_scaling_decomposition.csv").as_posix())

In [ ]:
# ---- Config (same capital list as notebook 14) ----
CAPITAL_IDS = {"HU110", "CZ010", "SK010", "PL911"}
ROBUST_PATH = OUT_DIR / "core_scaling_robustness.csv"
# ----------------------------------------------------

def fit_three(data, with_country_fe):
    """Fit ln Y, ln E, ln P on ln N and return slopes with the identity check."""
    tail = f" + C({COL_COUNTRY})" if with_country_fe else ""
    slopes, errors = {}, {}
    for label, dep in [("beta_Y", "ln_y"), ("beta_E", "ln_e"), ("beta_P", "ln_p")]:
        res = smf.ols(f"{dep} ~ ln_n{tail}", data=data).fit(cov_type=COV_TYPE)
        slopes[label] = res.params["ln_n"]
        errors[label] = res.bse["ln_n"]
    return {**slopes,
            **{"se_" + k: v for k, v in errors.items()},
            "n": len(data),
            "identity_diff": slopes["beta_Y"] - slopes["beta_E"] - slopes["beta_P"]}

# Build the samples: pooled (with country fixed effects) and per country
no_cap = df[~df[COL_ID].isin(CAPITAL_IDS)]
print(f"Capitals removed: {len(df) - len(no_cap)} (expected {len(CAPITAL_IDS)})")

samples = [("pooled_all", df, True), ("pooled_no_capitals", no_cap, True)]
for country in sorted(df[COL_COUNTRY].unique()):
    samples.append((f"{country}_all", df[df[COL_COUNTRY] == country], False))
    samples.append((f"{country}_no_capitals", no_cap[no_cap[COL_COUNTRY] == country], False))

rows = [{"sample": name, **fit_three(data, fe)} for name, data, fe in samples]
robust = pd.DataFrame(rows)

show = ["sample", "beta_Y", "beta_E", "beta_P", "n", "identity_diff"]
print(robust[show].round(4).to_string(index=False))

robust.to_csv(ROBUST_PATH, index=False)
print("Saved:", ROBUST_PATH.as_posix())

In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests

# ---- Config (change names and paths here only) ----
API_BASE = "https://bdl.stat.gov.pl/api/v1"
GROUP_ID = "G652"                         # NSP 2021 - economic activity of the population
NAME_MUST_CONTAIN = ["pracujący", "statusu zatrudnienia"]   # lower-case match
RAW_DIR = Path("data/raw/census_2021/pl")
SUBJECTS_CSV = RAW_DIR / "bdl_subjects_G652.csv"
PAGE_SIZE = 100
REQUEST_PAUSE_S = 0.3                     # anonymous limit: 5 requests per second
TIMEOUT_S = 30
# ----------------------------------------------------

RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_subjects(parent_id):
    """Download all subjects under parent_id (paged JSON:API response)."""
    rows, page, total = [], 0, None
    while total is None or len(rows) < total:
        resp = requests.get(
            f"{API_BASE}/subjects",
            params={"parent-id": parent_id, "format": "jsonapi",
                    "lang": "pl", "page": page, "page-size": PAGE_SIZE},
            timeout=TIMEOUT_S,
        )
        resp.raise_for_status()
        body = resp.json()
        total = body["meta"]["totalRecords"]
        data = body["data"]
        if not data:
            break
        for item in data:
            attrs = item["attributes"]
            rows.append({
                "id": item["id"],
                "name": attrs["name"],
                "has_variables": attrs["hasVariables"],
                "levels": ",".join(str(x) for x in attrs.get("levels", [])),
            })
        page += 1
        time.sleep(REQUEST_PAUSE_S)
    return pd.DataFrame(rows)

# Idempotent: skip the download when the file already exists
if SUBJECTS_CSV.exists():
    subjects = pd.read_csv(SUBJECTS_CSV, dtype={"levels": str})
    print("Loaded from disk:", SUBJECTS_CSV.as_posix())
else:
    try:
        subjects = fetch_subjects(GROUP_ID)
        subjects.to_csv(SUBJECTS_CSV, index=False)
        print("Downloaded and saved:", SUBJECTS_CSV.as_posix())
    except Exception as exc:                  # print only the class name (no paths)
        raise RuntimeError(
            f"BDL request failed ({type(exc).__name__}). Check the subgroup by hand "
            "on the BDL site (NSP 2021 - AKTYWNOSC EKONOMICZNA LUDNOSCI) "
            "and save the list to the file above."
        ) from None

print(f"Subjects in {GROUP_ID}: {len(subjects)}")

# Keep subjects whose name contains all required text parts
names = subjects["name"].str.lower()
mask = pd.Series(True, index=subjects.index)
for part in NAME_MUST_CONTAIN:
    mask &= names.str.contains(part, regex=False)
found = subjects[mask]
print(found.to_string(index=False))

In [ ]:
# ---- Config (change names and paths here only) ----
SUBJECT_ID = "P4312"        # employed, by sex / status / urban-rural, resident population
VARIABLES_CSV = RAW_DIR / f"bdl_variables_{SUBJECT_ID}.csv"
# (API_BASE, RAW_DIR, PAGE_SIZE, REQUEST_PAUSE_S, TIMEOUT_S come from the previous cell)
# ----------------------------------------------------

def fetch_variables(subject_id):
    """Download all variables of one BDL subject (paged JSON:API response)."""
    rows, page, total = [], 0, None
    while total is None or len(rows) < total:
        resp = requests.get(
            f"{API_BASE}/variables",
            params={"subject-id": subject_id, "format": "jsonapi",
                    "lang": "pl", "page": page, "page-size": PAGE_SIZE},
            timeout=TIMEOUT_S,
        )
        resp.raise_for_status()
        body = resp.json()
        total = body["meta"]["totalRecords"]
        data = body["data"]
        if not data:
            break
        for item in data:
            row = {"id": item["id"]}
            row.update(item["attributes"])      # keep every attribute the API returns
            rows.append(row)
        page += 1
        time.sleep(REQUEST_PAUSE_S)
    return pd.DataFrame(rows)

# Idempotent: skip the download when the file already exists
if VARIABLES_CSV.exists():
    variables = pd.read_csv(VARIABLES_CSV)
    print("Loaded from disk:", VARIABLES_CSV.as_posix())
else:
    try:
        variables = fetch_variables(SUBJECT_ID)
        variables.to_csv(VARIABLES_CSV, index=False)
        print("Downloaded and saved:", VARIABLES_CSV.as_posix())
    except Exception as exc:                  # print only the class name (no paths)
        raise RuntimeError(
            f"BDL request failed ({type(exc).__name__}). Open the subgroup by hand "
            "on the BDL site and save the variable list to the file above."
        ) from None

print(f"Variables in {SUBJECT_ID}: {len(variables)}")
print("Columns:", list(variables.columns))

# Show the id and the dimension-name columns (n1, n2, ...) when they exist
name_cols = [c for c in variables.columns if c[0] == "n" and c[1:].isdigit()]
show_cols = ["id"] + name_cols
print(variables[show_cols].to_string(index=False))

In [ ]:
import json

# ---- Config (change names and paths here only) ----
SEX_LABEL = "ogółem"                    # both sexes
STATUS_LABEL = "ogółem"                 # all employment statuses
RESIDENCE_LABELS = {"total": "ogółem", "urban": "w miastach", "rural": "na wsi"}
YEAR = 2021
UNIT_LEVEL = 5                          # 5 = powiat in the BDL API
# (API_BASE, RAW_DIR, PAGE_SIZE, REQUEST_PAUSE_S, TIMEOUT_S, variables come from earlier cells)
# ----------------------------------------------------

# Pick the three variables: total / urban / rural, all sexes, all statuses
sel = variables[(variables["n2"] == SEX_LABEL) & (variables["n3"] == STATUS_LABEL)]
sel = sel.set_index("n1")
var_ids = {key: int(sel.loc[label, "id"]) for key, label in RESIDENCE_LABELS.items()}
print("Selected variables:", var_ids)
print(sel.loc[list(RESIDENCE_LABELS.values()),
              ["id", "level", "measureUnitName"]].to_string())

def page_path(var_id, page):
    return RAW_DIR / f"bdl_data_{var_id}_level{UNIT_LEVEL}_page{page}.json"

def download_pages(var_id):
    """Download all pages of one variable; skip pages that are already on disk."""
    page, total = 0, None
    while True:
        path = page_path(var_id, page)
        if path.exists():
            body = json.loads(path.read_text(encoding="utf-8"))
        else:
            try:
                resp = requests.get(
                    f"{API_BASE}/data/by-variable/{var_id}",
                    params={"format": "jsonapi", "year": YEAR,
                            "unit-level": UNIT_LEVEL, "page": page,
                            "page-size": PAGE_SIZE},
                    timeout=TIMEOUT_S,
                )
                resp.raise_for_status()
                body = resp.json()
            except Exception as exc:          # print only the class name (no paths)
                raise RuntimeError(
                    f"BDL request failed for variable {var_id}, page {page} "
                    f"({type(exc).__name__}). Rerun the cell: finished pages are skipped."
                ) from None
            path.write_text(json.dumps(body, ensure_ascii=False), encoding="utf-8")
            time.sleep(REQUEST_PAUSE_S)
        total = body["meta"]["totalRecords"]
        page += 1
        if page * PAGE_SIZE >= total or not body["data"]:
            return page, total

for key, var_id in var_ids.items():
    n_pages, total = download_pages(var_id)
    print(f"{key}: variable {var_id}, {n_pages} pages, totalRecords = {total}")

# Show the structure of the first page so the parser can be written exactly
first = json.loads(page_path(var_ids["total"], 0).read_text(encoding="utf-8"))
print("Top-level keys:", list(first.keys()))
print("Keys of data[0]:", list(first["data"][0].keys()))
print("Keys of data[0]['attributes']:", list(first["data"][0]["attributes"].keys()))
print("data[0] (first 400 characters):")
print(json.dumps(first["data"][0], ensure_ascii=False)[:400])

In [ ]:
# ---- Config (change names and paths here only) ----
EMPLOYED_CSV = RAW_DIR / "pl_resident_employed_powiat_2021.csv"
# (var_ids, page_path, UNIT_LEVEL, YEAR, PAGE_SIZE come from the previous cell)
# ----------------------------------------------------

def read_variable(var_id, col_name):
    """Read all saved pages of one variable into a table (one row per unit)."""
    rows, page = [], 0
    while page_path(var_id, page).exists():
        body = json.loads(page_path(var_id, page).read_text(encoding="utf-8"))
        for item in body["data"]:
            vals = [v for v in item["attributes"]["values"] if v["year"] == str(YEAR)]
            val = vals[0].get("val") if vals else None
            attr = vals[0].get("attrId") if vals else None
            rows.append({"unit_id": item["id"],
                         "unit_name": item["attributes"]["name"],
                         col_name: val,
                         f"attr_{col_name}": attr})
        page += 1
    return pd.DataFrame(rows)

if EMPLOYED_CSV.exists():
    employed = pd.read_csv(EMPLOYED_CSV, dtype={"unit_id": str})
    print("Loaded from disk:", EMPLOYED_CSV.as_posix())
else:
    parts = [read_variable(var_ids["total"], "employed_total"),
             read_variable(var_ids["urban"], "employed_urban"),
             read_variable(var_ids["rural"], "employed_rural")]
    employed = parts[0]
    for part in parts[1:]:
        employed = employed.merge(part.drop(columns="unit_name"),
                                  on="unit_id", how="outer", validate="1:1")
    employed.to_csv(EMPLOYED_CSV, index=False)
    print("Built and saved:", EMPLOYED_CSV.as_posix())

print("Powiats:", len(employed), "| unique ids:", employed["unit_id"].nunique())
value_cols = ["employed_total", "employed_urban", "employed_rural"]
print("Missing values per column:\n", employed[value_cols].isna().sum().to_string())
attr_cols = [c for c in employed.columns if c.startswith("attr_")]
for c in attr_cols:
    print(c, employed[c].value_counts(dropna=False).to_dict())

# Check 1: urban + rural = total per powiat
diff = employed["employed_urban"] + employed["employed_rural"] - employed["employed_total"]
print("Powiats where urban + rural != total:", int((diff.abs() > 0).sum()),
      "| max abs difference:", diff.abs().max())

# Check 2: national total (informative only, definitions may differ)
print("Sum of employed_total over powiats:", int(employed["employed_total"].sum()))

In [ ]:
# ---- Config (change names and paths here only) ----
UNIT_LEVELS = {"powiat": 5, "subregion": 4}     # 5 = powiat, 4 = podregion (NUTS3)
# (API_BASE, RAW_DIR, PAGE_SIZE, REQUEST_PAUSE_S, TIMEOUT_S come from earlier cells)
# ----------------------------------------------------

def units_page_path(level, page):
    return RAW_DIR / f"bdl_units_level{level}_page{page}.json"

def download_units(level):
    """Download all units of one level; skip pages that are already on disk."""
    page, total = 0, None
    while True:
        path = units_page_path(level, page)
        if path.exists():
            body = json.loads(path.read_text(encoding="utf-8"))
        else:
            try:
                resp = requests.get(
                    f"{API_BASE}/units",
                    params={"level": level, "format": "jsonapi", "lang": "pl",
                            "page": page, "page-size": PAGE_SIZE},
                    timeout=TIMEOUT_S,
                )
                resp.raise_for_status()
                body = resp.json()
            except Exception as exc:          # print only the class name (no paths)
                raise RuntimeError(
                    f"BDL request failed for level {level}, page {page} "
                    f"({type(exc).__name__}). Rerun the cell: finished pages are skipped."
                ) from None
            path.write_text(json.dumps(body, ensure_ascii=False), encoding="utf-8")
            time.sleep(REQUEST_PAUSE_S)
        total = body["meta"]["totalRecords"]
        page += 1
        if page * PAGE_SIZE >= total or not body["data"]:
            return page, total

for label, level in UNIT_LEVELS.items():
    n_pages, total = download_units(level)
    print(f"{label} (level {level}): {n_pages} pages, totalRecords = {total}")

# Show the structure of one record per level so the parser can be written exactly
for label, level in UNIT_LEVELS.items():
    first = json.loads(units_page_path(level, 0).read_text(encoding="utf-8"))
    print(f"\n{label}: keys of data[0]['attributes'] =",
          list(first["data"][0]["attributes"].keys()))
    print(json.dumps(first["data"][0], ensure_ascii=False)[:500])

In [ ]:
# ---- Config (change names and paths here only) ----
GMINA_NUTS3_CSV = Path("data/interim/pl_gmina_nuts3.csv")
NUTS3_CSV = Path("results/core_scaling_decomposition/pl_resident_employed_nuts3_2021.csv")
VOIV_SLICE = slice(2, 4)         # voivodeship digits inside the 12-digit BDL unit id
POWIAT_SLICE = slice(7, 9)       # powiat digits inside the 12-digit BDL unit id
VALUE_COLS = ["employed_total", "employed_urban", "employed_rural"]
COUNTRY_CODE_PL = "PL"
# (EMPLOYED_CSV, MASTER_PATH, COL_ID, COL_COUNTRY come from earlier cells)
# ----------------------------------------------------

NUTS3_CSV.parent.mkdir(parents=True, exist_ok=True)

employed = pd.read_csv(EMPLOYED_CSV, dtype={"unit_id": str})

# 4-digit TERYT powiat code from the BDL unit id (derived from one example, checked below)
employed["powiat_code"] = (employed["unit_id"].str[VOIV_SLICE]
                           + employed["unit_id"].str[POWIAT_SLICE])
print("Powiats:", len(employed), "| unique TERYT codes:", employed["powiat_code"].nunique())

# Powiat -> NUTS3 map from the gmina table; each powiat must have exactly one NUTS3
gmina = pd.read_csv(GMINA_NUTS3_CSV, dtype=str)
pmap = gmina[["powiat_code", "nuts3_code"]].drop_duplicates()
n_nuts_per_powiat = pmap.groupby("powiat_code")["nuts3_code"].nunique()
print("Powiats with more than one NUTS3:", int((n_nuts_per_powiat > 1).sum()))

merged = employed.merge(pmap, on="powiat_code", how="left", validate="m:1")
unmatched = merged[merged["nuts3_code"].isna()]
print("Powiats without NUTS3:", len(unmatched))
if len(unmatched):
    print(unmatched[["unit_id", "unit_name", "powiat_code"]].to_string(index=False))

# Aggregate to NUTS3 (podregion)
nuts3 = merged.groupby("nuts3_code", as_index=False)[VALUE_COLS].sum()
print("NUTS3 regions:", len(nuts3))
print("Sums equal before/after aggregation:",
      bool((nuts3[VALUE_COLS].sum() == merged[VALUE_COLS].sum()).all()))

# Compare with the Polish NUTS3 ids of the master table
master_ids = pd.read_parquet(MASTER_PATH, columns=[COL_ID, COL_COUNTRY])
master_pl = set(master_ids.loc[master_ids[COL_COUNTRY] == COUNTRY_CODE_PL, COL_ID])
ours = set(nuts3["nuts3_code"])
print("In master but not in our table:", sorted(master_pl - ours))
print("In our table but not in master:", sorted(ours - master_pl))

nuts3.to_csv(NUTS3_CSV, index=False)
print("Saved:", NUTS3_CSV.as_posix())
print(nuts3.nlargest(3, "employed_total").to_string(index=False))

In [ ]:
import time
import requests
import pandas as pd
from pathlib import Path

# ---- Config (change names and paths here only) ----
CZ_RAW_DIR = Path("data/raw/census_2021/cz")
DATASETS = {
    "sldb2021_aktivita_pohlavi.csv":
        "https://csu.gov.cz/docs/107508/ce2847c2-5f85-399c-f905-3bd1eff51cc0/sldb2021_aktivita_pohlavi.csv?version=1.0",
    "sldb2021_aktivita_pohlavi-metadata.json":
        "https://csu.gov.cz/docs/107508/d23c842d-01a7-5a47-0e9b-f7d5e6ccd89b/sldb2021_aktivita_pohlavi-metadata.json?version=1.0",
}
TIMEOUT_S = 120
CHUNK_BYTES = 1 << 20
MAX_ATTEMPTS = 8
BACKOFF_S = 3
MAX_CATEGORIES_SHOWN = 20
# ----------------------------------------------------

CZ_RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, path):
    """Idempotent download with retries; resumes the .part file with a Range request."""
    if path.exists() and path.stat().st_size > 0:
        print("Already on disk:", path.as_posix())
        return
    part = path.with_name(path.name + ".part")
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        have = part.stat().st_size if part.exists() else 0
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, stream=True, headers=headers, timeout=TIMEOUT_S) as resp:
                if resp.status_code == 416:       # range beyond the end: file is complete
                    break
                resp.raise_for_status()
                if resp.status_code == 206:       # server resumes: append
                    mode = "ab"
                else:                             # server restarts: write from scratch
                    mode, have = "wb", 0
                with open(part, mode) as fh:
                    for chunk in resp.iter_content(CHUNK_BYTES):
                        fh.write(chunk)
            break                                  # finished without an error
        except Exception as exc:                   # print only the class name (no paths)
            last_error = type(exc).__name__
            got = part.stat().st_size / 1e6 if part.exists() else 0.0
            print(f"Attempt {attempt}/{MAX_ATTEMPTS} failed ({last_error}), "
                  f"{got:.1f} MB saved so far; retrying.")
            time.sleep(BACKOFF_S * attempt)
    else:
        raise RuntimeError(
            f"Download failed for {path.name} ({last_error}). Rerun the cell (the partial "
            "file is kept and resumed), or download it by hand from the CSU open-data page "
            "of the 2021 census and save it to the folder above."
        ) from None
    part.replace(path)
    print("Downloaded:", path.as_posix(), f"({path.stat().st_size / 1e6:.1f} MB)")

for name, url in DATASETS.items():
    download_file(url, CZ_RAW_DIR / name)

# Inspect the structure of the CSV (no assumptions about the columns)
csv_path = CZ_RAW_DIR / "sldb2021_aktivita_pohlavi.csv"
cz_act = pd.read_csv(csv_path, dtype=str)
print("Rows:", len(cz_act))
print("Columns:", list(cz_act.columns))

for col in cz_act.columns:
    if col.endswith("_txt") and col != "uzemi_txt":
        vals = cz_act[col].value_counts(dropna=False)
        print(f"\n{col} ({len(vals)} categories):")
        print(vals.head(MAX_CATEGORIES_SHOWN).to_string())

if "uzemi_typ" in cz_act.columns:
    print("\nuzemi_typ:")
    print(cz_act["uzemi_typ"].value_counts(dropna=False).to_string())

In [ ]:
# ---- Config (change names and paths here only) ----
LABEL_EMPLOYED = "Zaměstnaní"
LABEL_LABOUR_FORCE = "Pracovní síla"
LABEL_UNEMPLOYED = "Nezaměstnaní"
LEVEL_KRAJ = "kraj"
LEVEL_COUNTRY = "stát"
CZ_KRAJ_CSV = Path("results/core_scaling_decomposition/cz_resident_employed_kraj_2021.csv")
# (cz_act comes from the previous cell)
# ----------------------------------------------------

CZ_KRAJ_CSV.parent.mkdir(parents=True, exist_ok=True)

total_sex = cz_act["pohlavi_txt"].isna()          # empty sex = both sexes together
base = cz_act[total_sex].copy()
base["hodnota"] = pd.to_numeric(base["hodnota"])

def pick(label, level):
    """Rows of one activity category at one territorial level (both sexes)."""
    sub = base[(base["ekonaktiv_txt"] == label) & (base["uzemi_typ"] == level)]
    return sub.set_index("uzemi_kod")["hodnota"]

kraj_info = (base[base["uzemi_typ"] == LEVEL_KRAJ][["uzemi_kod", "uzemi_txt"]]
             .drop_duplicates().set_index("uzemi_kod"))
emp = pick(LABEL_EMPLOYED, LEVEL_KRAJ)
cz_kraj = kraj_info.join(emp.rename("employed_total"))
cz_kraj["labour_force"] = pick(LABEL_LABOUR_FORCE, LEVEL_KRAJ)
cz_kraj["unemployed"] = pick(LABEL_UNEMPLOYED, LEVEL_KRAJ)
cz_kraj = cz_kraj.reset_index()
print(cz_kraj.to_string(index=False))

# Check 1: 14 krajs
print("\nNumber of krajs:", len(cz_kraj))

# Check 2: sum of krajs equals the country row
country_total = pick(LABEL_EMPLOYED, LEVEL_COUNTRY).sum()
print("Sum of krajs:", int(cz_kraj["employed_total"].sum()),
      "| country row:", int(country_total))

# Check 3: labour force - unemployed = employed
diff = cz_kraj["labour_force"] - cz_kraj["unemployed"] - cz_kraj["employed_total"]
print("Max |labour force - unemployed - employed|:", diff.abs().max())

cz_kraj.to_csv(CZ_KRAJ_CSV, index=False)
print("Saved:", CZ_KRAJ_CSV.as_posix())

In [ ]:
import json
import unicodedata
from pathlib import Path
import pandas as pd

# ---- config ----
GISCO_GEOJSON = Path("data/raw/nuts3/NUTS_RG_03M_2024_4326.geojson")
CZ_KRAJ_CSV = Path("results/core_scaling_decomposition/cz_resident_employed_kraj_2021.csv")
OUT_DIR = Path("results/core_scaling_decomposition")
OUT_MAP = OUT_DIR / "cz_kraj_nuts3_map.csv"
OUT_EMPLOYED = OUT_DIR / "cz_resident_employed_nuts3_2021.csv"
GISCO_COL_ID, GISCO_COL_NAME = "NUTS_ID", "NUTS_NAME"
GISCO_COL_LEVEL, GISCO_COL_CNTR = "LEVL_CODE", "CNTR_CODE"
COUNTRY, NUTS_LEVEL = "CZ", 3
EXPECTED_N = 14
# ----------------

def norm(s):
    # case-folded, whitespace-collapsed name used only for matching
    return " ".join(str(s).strip().casefold().split())

# read NUTS3 names of the country from the GISCO file
with open(GISCO_GEOJSON, encoding="utf-8") as f:
    feats = json.load(f)["features"]
gisco = pd.DataFrame([ft["properties"] for ft in feats])
gisco = gisco[(gisco[GISCO_COL_CNTR] == COUNTRY) & (gisco[GISCO_COL_LEVEL] == NUTS_LEVEL)]
gisco = gisco[[GISCO_COL_ID, GISCO_COL_NAME]].rename(
    columns={GISCO_COL_ID: "nuts3_code", GISCO_COL_NAME: "nuts3_name"})
gisco["key"] = gisco["nuts3_name"].map(norm)

cz = pd.read_csv(CZ_KRAJ_CSV)
cz["key"] = cz["uzemi_txt"].map(norm)

merged = cz.merge(gisco, on="key", how="outer", indicator=True)
unmatched = merged[merged["_merge"] != "both"]

if len(unmatched) == 0 and len(merged) == EXPECTED_N:
    mapping = merged[["uzemi_kod", "uzemi_txt", "nuts3_code", "nuts3_name"]].sort_values("nuts3_code")
    mapping.to_csv(OUT_MAP, index=False)
    out = merged.drop(columns=["key", "_merge", "nuts3_name"]).sort_values("nuts3_code")
    out.to_csv(OUT_EMPLOYED, index=False)
    print(f"Matched {len(mapping)}/{EXPECTED_N}; saved {OUT_MAP} and {OUT_EMPLOYED}")
    print(mapping.to_string(index=False))
else:
    print("NOT all names matched; nothing saved. Unmatched rows:")
    print(unmatched[["uzemi_kod", "uzemi_txt", "nuts3_code", "nuts3_name", "_merge"]].to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ---- config ----
MASTER_PATH = Path("data/processed/nuts3_master_table.parquet")
RES_DIR = Path("results/core_scaling_decomposition")
YEAR = 2021
COUNTRIES = {
    "CZ": {"od": Path("data/processed/05_gravity_cz/od_observed.csv"),
           "r": RES_DIR / "cz_resident_employed_nuts3_2021.csv"},
    "PL": {"od": Path("data/processed/05b_gravity_pl/od_observed.csv"),
           "r": RES_DIR / "pl_resident_employed_nuts3_2021.csv"},
}
COL_ID, COL_YEAR, COL_COUNTRY = "nuts3_id", "year", "country_code"
COL_E, COL_N = "employment", "population"
COL_R_CODE, COL_R_VALUE = "nuts3_code", "employed_total"
OUT_PATH = RES_DIR / "commuting_balance_2021.csv"
# ----------------

master = pd.read_parquet(MASTER_PATH)
master = master[master[COL_YEAR] == YEAR].set_index(COL_ID)

rows = []
for cc, cfg in COUNTRIES.items():
    od = pd.read_csv(cfg["od"], index_col=0)       # first column = origin region code
    od.index = od.index.astype(str)
    od = od.loc[od.index, od.index]                # align rows and columns on the same codes
    m = od.to_numpy(dtype=float)
    diag = np.diag(m).copy()
    np.fill_diagonal(m, 0.0)                       # remove the diagonal for I and X
    out = pd.DataFrame({
        "nuts3_id": od.index,
        "country_code": cc,
        "X_out": m.sum(axis=1),                    # row sum = out-commuters
        "I_in": m.sum(axis=0),                     # column sum = in-commuters
        "od_diag": diag,
        "od_rowsum_with_diag": m.sum(axis=1) + diag,
    })
    r = pd.read_csv(cfg["r"]).rename(columns={COL_R_CODE: "nuts3_id", COL_R_VALUE: "R_census"})
    out = out.merge(r[["nuts3_id", "R_census"]], on="nuts3_id", how="left")
    rows.append(out)

bal = pd.concat(rows, ignore_index=True)
bal = bal.merge(master[[COL_E, COL_N]].rename(columns={COL_E: "E_eurostat", COL_N: "population_2021"}),
                left_on="nuts3_id", right_index=True, how="left")
bal["E_from_identity"] = bal["R_census"] + bal["I_in"] - bal["X_out"]
bal["ratio_E_identity"] = bal["E_eurostat"] / bal["E_from_identity"]
bal["ratio_rowsum_R"] = bal["od_rowsum_with_diag"] / bal["R_census"]

bal.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, "| rows:", len(bal), "| missing E:", int(bal["E_eurostat"].isna().sum()),
      "| missing R:", int(bal["R_census"].isna().sum()))
print(bal.groupby("country_code")[["ratio_E_identity", "ratio_rowsum_R"]].describe().T.round(3).to_string())

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config ----
BAL_PATH = Path("results/core_scaling_decomposition/commuting_balance_2021.csv")
E_UNIT_FACTOR = 1000   # assumption to verify: Eurostat employment in thousand persons
# ----------------

bal = pd.read_csv(BAL_PATH)
bal["E_persons"] = bal["E_eurostat"] * E_UNIT_FACTOR
bal["ratio_E_identity_scaled"] = bal["E_persons"] / bal["E_from_identity"]
bal.to_csv(BAL_PATH, index=False)

print("Saved:", BAL_PATH)
print(bal.groupby("country_code")["ratio_E_identity_scaled"].describe().round(3).to_string())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---- config ----
BAL_PATH = Path("results/core_scaling_decomposition/commuting_balance_2021.csv")
OUT_PATH = Path("results/core_scaling_decomposition/delta_comm_2021.csv")
COL_E, COL_R, COL_N, COL_COUNTRY = "E_persons", "R_census", "population_2021", "country_code"
COV_TYPE = "HC3"
ALPHA = 0.05
# ----------------

bal = pd.read_csv(BAL_PATH)
d = pd.DataFrame({
    "country_code": bal[COL_COUNTRY],
    "ln_n": np.log(bal[COL_N]),
    "ln_e": np.log(bal[COL_E]),
    "ln_r": np.log(bal[COL_R]),
})
d["ln_er"] = d["ln_e"] - d["ln_r"]   # ln(E/R); its slope equals beta_E - beta_R

def fit(data, dep, with_fe):
    rhs = "ln_n + C(country_code)" if with_fe else "ln_n"
    res = smf.ols(f"{dep} ~ {rhs}", data=data).fit(cov_type=COV_TYPE)
    lo, hi = res.conf_int(alpha=ALPHA).loc["ln_n"]
    return {"beta": res.params["ln_n"], "se": res.bse["ln_n"], "ci_low": lo, "ci_high": hi, "n": int(res.nobs)}

rows = []
samples = [("pooled_CZ_PL", d, True)] + [(cc, d[d["country_code"] == cc], False) for cc in sorted(d["country_code"].unique())]
for name, data, fe in samples:
    for label, dep in [("beta_E", "ln_e"), ("beta_R", "ln_r"), ("delta_comm", "ln_er")]:
        rows.append({"sample": name, "quantity": label, **fit(data, dep, fe)})

out = pd.DataFrame(rows)
out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
print(out.round(4).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---- config ----
RES_DIR = Path("results/core_scaling_decomposition")
BAL_PATH = RES_DIR / "commuting_balance_2021.csv"
DIRS = {"CZ": Path("data/processed/05_gravity_cz"), "PL": Path("data/processed/05b_gravity_pl")}
MODELS = ["gravity_power", "gravity_exp", "radiation"]
COL_TILE = "tile_ID"
OUT_DELTA = RES_DIR / "null_model_delta_2021.csv"
OUT_EHAT = RES_DIR / "null_model_Ehat_2021.csv"
COV_TYPE = "HC3"
# ----------------

bal = pd.read_csv(BAL_PATH).set_index("nuts3_id")

# per model: list of per-realization frames (all countries stacked)
per_real = {m: [] for m in MODELS}
checks = []
for cc, folder in DIRS.items():
    ids = pd.read_csv(folder / "tessellation_outflows.csv")[COL_TILE].astype(str).tolist()
    od_ids = pd.read_csv(folder / "od_observed.csv", index_col=0).index.astype(str).tolist()
    assert ids == od_ids, f"{cc}: region order differs between tessellation and observed matrix"
    sub = bal.loc[ids]
    for m in MODELS:
        arr = np.load(folder / f"realizations_{m}.npy")           # (n_real, N, N)
        x_hat = arr.sum(axis=2)                                    # (n_real, N) out-commuters
        i_hat = arr.sum(axis=1)                                    # (n_real, N) in-commuters
        checks.append({"country": cc, "model": m,
                       "max_rel_diff_Xhat_vs_X": float(np.max(np.abs(x_hat.mean(0) - sub["X_out"].to_numpy()) / sub["X_out"].to_numpy()))})
        for k in range(arr.shape[0]):
            per_real[m].append(pd.DataFrame({
                "nuts3_id": ids, "country_code": cc, "model": m, "realization": k,
                "R": sub["R_census"].to_numpy(), "N": sub["population_2021"].to_numpy(),
                "E_hat": sub["R_census"].to_numpy() + i_hat[k] - x_hat[k]}))

print(pd.DataFrame(checks).round(4).to_string(index=False))

all_df = pd.concat([f for m in MODELS for f in per_real[m]], ignore_index=True)
assert (all_df["E_hat"] > 0).all(), "non-positive E_hat found"

rows = []
for (m, k), g in all_df.groupby(["model", "realization"]):
    d = pd.DataFrame({"country_code": g["country_code"], "ln_n": np.log(g["N"]),
                      "dep": np.log(g["E_hat"] / g["R"])})
    res = smf.ols("dep ~ ln_n + C(country_code)", data=d).fit(cov_type=COV_TYPE)
    rows.append({"model": m, "realization": k, "delta_null": res.params["ln_n"], "se": res.bse["ln_n"]})
delta = pd.DataFrame(rows)
delta.to_csv(OUT_DELTA, index=False)

ehat_mean = all_df.groupby(["model", "nuts3_id"], as_index=False)["E_hat"].mean()
ehat_mean.to_csv(OUT_EHAT, index=False)

print("Saved:", OUT_DELTA, "and", OUT_EHAT)
summary = delta.groupby("model")["delta_null"].agg(["mean", "std", "min", "max"]).round(4)
print(summary.to_string())
obs = pd.read_csv(RES_DIR / "delta_comm_2021.csv")
print("Observed delta_comm (pooled):",
      obs[(obs["sample"] == "pooled_CZ_PL") & (obs["quantity"] == "delta_comm")][["beta", "ci_low", "ci_high"]].round(4).to_dict("records"))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---- config ----
MASTER_PATH = Path("data/processed/nuts3_master_table.parquet")
RES_DIR = Path("results/core_scaling_decomposition")
BAL_PATH = RES_DIR / "commuting_balance_2021.csv"
DIRS = {"CZ": Path("data/processed/05_gravity_cz"), "PL": Path("data/processed/05b_gravity_pl")}
MODELS = ["gravity_power", "gravity_exp", "radiation"]
YEAR = 2021
COL_ID, COL_YEAR, COL_Y = "nuts3_id", "year", "gdp_total"
COL_N = "population_2021"
REALIZATION = 0                      # all realizations are identical (checked), use the first
COV_TYPE, ALPHA = "HC3", 0.05
OUT_REGION = RES_DIR / "excess_attraction_2021.csv"
OUT_REG = RES_DIR / "excess_attraction_regression_2021.csv"
# ----------------

bal = pd.read_csv(BAL_PATH).set_index("nuts3_id")
master = pd.read_parquet(MASTER_PATH)
master = master[master[COL_YEAR] == YEAR].set_index(COL_ID)

frames = []
for cc, folder in DIRS.items():
    od = pd.read_csv(folder / "od_observed.csv", index_col=0)
    od.index = od.index.astype(str)
    ids = od.index.tolist()
    T = od.loc[ids, ids].to_numpy(dtype=float)
    np.fill_diagonal(T, 0.0)                         # only inter-regional flows (i != j)
    D = T.sum(axis=0)                                # actual inflow per destination
    assert (D > 0).all(), f"{cc}: a region has zero observed inflow"
    q = T / D                                        # origin shares q_ij (columns sum to 1)
    qlogq = q * np.log(np.where(q > 0, q, 1.0))      # log(1) = 0, so zero shares add 0
    H = -qlogq.sum(axis=0)                           # origin entropy per destination
    out = pd.DataFrame({"nuts3_id": ids, "country_code": cc, "D_obs": D,
                        "origin_entropy": H, "R_eff": np.exp(H)})
    for m in MODELS:
        arr = np.load(folder / f"realizations_{m}.npy", mmap_mode="r")
        That = np.array(arr[REALIZATION], dtype=float)
        np.fill_diagonal(That, 0.0)
        out[f"D_hat_{m}"] = That.sum(axis=0)
    frames.append(out)

reg = pd.concat(frames, ignore_index=True).set_index("nuts3_id")
for m in MODELS:
    assert (reg[f"D_hat_{m}"] > 0).all(), f"non-positive modelled inflow: {m}"
    reg[f"A_{m}"] = np.log(reg["D_obs"] / reg[f"D_hat_{m}"])

# GDP scaling residual (2021, country fixed effects), same sample
reg["gdp"] = master.loc[reg.index, COL_Y]
reg["N"] = bal.loc[reg.index, COL_N]
reg["ln_y"], reg["ln_n"] = np.log(reg["gdp"]), np.log(reg["N"])
base = smf.ols("ln_y ~ ln_n + C(country_code)", data=reg).fit(cov_type=COV_TYPE)
reg["eta_Y"] = base.resid
reg["ln_R_eff"] = np.log(reg["R_eff"])

rows = []
for x in [f"A_{m}" for m in MODELS] + ["ln_R_eff"]:
    res = smf.ols(f"eta_Y ~ {x} + C(country_code)", data=reg).fit(cov_type=COV_TYPE)
    lo, hi = res.conf_int(alpha=ALPHA).loc[x]
    rows.append({"regressor": x, "gamma": res.params[x], "se": res.bse[x],
                 "ci_low": lo, "ci_high": hi, "p": res.pvalues[x], "n": int(res.nobs)})
out_reg = pd.DataFrame(rows)

reg.reset_index().to_csv(OUT_REGION, index=False)
out_reg.to_csv(OUT_REG, index=False)
print("Saved:", OUT_REGION, "and", OUT_REG)
print(out_reg.round(4).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---- config ----
RES_DIR = Path("results/core_scaling_decomposition")
IN_PATH = RES_DIR / "excess_attraction_2021.csv"
OUT_PATH = RES_DIR / "excess_attraction_robustness_2021.csv"
CAPITALS_IN_SAMPLE = ["CZ010", "PL911"]
REGRESSORS = ["A_gravity_power", "A_gravity_exp", "A_radiation", "ln_R_eff"]
COV_TYPE, ALPHA = "HC3", 0.05
# ----------------

reg = pd.read_csv(IN_PATH)
samples = {"all": []}
for c in CAPITALS_IN_SAMPLE:
    samples[f"without_{c}"] = [c]
samples["without_both"] = list(CAPITALS_IN_SAMPLE)

rows = []
for name, drop in samples.items():
    d = reg[~reg["nuts3_id"].isin(drop)].copy()
    d["eta_Y"] = smf.ols("ln_y ~ ln_n + C(country_code)", data=d).fit().resid   # residual refit on this sample
    for x in REGRESSORS:
        res = smf.ols(f"eta_Y ~ {x} + C(country_code)", data=d).fit(cov_type=COV_TYPE)
        lo, hi = res.conf_int(alpha=ALPHA).loc[x]
        rows.append({"sample": name, "regressor": x, "gamma": res.params[x], "se": res.bse[x],
                     "ci_low": lo, "ci_high": hi, "p": res.pvalues[x], "n": int(res.nobs)})

out = pd.DataFrame(rows)
out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
print(out.round(4).to_string(index=False))

In [ ]:
import numpy as np

# ---- Config (change names and paths here only) ----
DIFF_PATH = OUT_DIR / "core_scaling_capital_diff.csv"
N_BOOT = 1000          # number of bootstrap draws
SEED = 42              # random seed for reproducibility
ALPHA = 0.05           # two-sided 95% percentile interval
LABELS = ["beta_Y", "beta_E", "beta_P"]
# ----------------------------------------------------

def pooled_slopes(data):
    """Pooled slopes with country fixed effects (plain OLS, same point estimates as fit_three)."""
    out = {}
    for label, dep in zip(LABELS, ["ln_y", "ln_e", "ln_p"]):
        res = smf.ols(f"{dep} ~ ln_n + C({COL_COUNTRY})", data=data).fit()
        out[label] = res.params["ln_n"]
    return out

def diff_all_vs_nocap(data):
    """Slope with all regions minus slope without capitals."""
    a = pooled_slopes(data)
    b = pooled_slopes(data[~data[COL_ID].isin(CAPITAL_IDS)])
    return {k: a[k] - b[k] for k in LABELS}

if DIFF_PATH.exists():
    diff_table = pd.read_csv(DIFF_PATH)
    print("Loaded existing:", DIFF_PATH.as_posix())
else:
    rng = np.random.default_rng(SEED)
    point = diff_all_vs_nocap(df)

    # Stratified bootstrap: resample regions with replacement within each country
    groups = [g for _, g in df.groupby(COL_COUNTRY)]
    draws = []
    for _ in range(N_BOOT):
        sample = pd.concat(
            [g.iloc[rng.integers(0, len(g), len(g))] for g in groups],
            ignore_index=True,
        )
        draws.append(diff_all_vs_nocap(sample))
    draws = pd.DataFrame(draws)

    diff_table = pd.DataFrame({
        "slope": LABELS,
        "diff_point": [point[k] for k in LABELS],
        "ci_low": [draws[k].quantile(ALPHA / 2) for k in LABELS],
        "ci_high": [draws[k].quantile(1 - ALPHA / 2) for k in LABELS],
        "share_positive": [(draws[k] > 0).mean() for k in LABELS],
        "n_boot": N_BOOT,
    })
    diff_table.to_csv(DIFF_PATH, index=False)
    print("Saved:", DIFF_PATH.as_posix())

print(diff_table.round(4).to_string(index=False))

In [ ]:
# ---- Config (change names and paths here only) ----
BALANCE_PATH = OUT_DIR / "commuting_balance_2021.csv"
OUT_PATH_2021 = OUT_DIR / "core_scaling_decomposition_2021.csv"
YEAR_2021 = 2021
COL_ID, COL_YEAR, COL_COUNTRY = "nuts3_id", "year", "country_code"
COL_N_2021, COL_E_2021 = "population_2021", "E_persons"   # from the balance file
COL_Y_MASTER = "gdp_total"                                 # from the master table
COV_TYPE = "HC3"
# ----------------------------------------------------

if OUT_PATH_2021.exists():
    out_2021 = pd.read_csv(OUT_PATH_2021)
    print("Loaded existing:", OUT_PATH_2021.as_posix())
else:
    # Population and employment of the 2021 commuting block (87 regions)
    bal = pd.read_csv(BALANCE_PATH, usecols=[COL_ID, COL_COUNTRY, COL_N_2021, COL_E_2021])

    # GDP of the same year from the master table (only the needed columns)
    gdp = pd.read_parquet(MASTER_PATH, columns=[COL_ID, COL_YEAR, COL_Y_MASTER])
    gdp = gdp[gdp[COL_YEAR] == YEAR_2021][[COL_ID, COL_Y_MASTER]]

    d21 = bal.merge(gdp, on=COL_ID, how="left")
    print(f"Regions in balance file: {len(bal)}, with GDP: {d21[COL_Y_MASTER].notna().sum()}")

    # One common sample: all three variables present and positive
    need = [COL_N_2021, COL_E_2021, COL_Y_MASTER]
    d21 = d21[d21[need].notna().all(axis=1) & (d21[need] > 0).all(axis=1)].copy()
    print(f"Common sample: {len(d21)}")
    print(d21[COL_COUNTRY].value_counts().to_string())

    # Logs; ln P = ln Y - ln E
    d21["ln_n"] = np.log(d21[COL_N_2021])
    d21["ln_y"] = np.log(d21[COL_Y_MASTER])
    d21["ln_e"] = np.log(d21[COL_E_2021])
    d21["ln_p"] = d21["ln_y"] - d21["ln_e"]

    # Same specification as the 2023 block: ln N + country fixed effects, HC3
    rows = []
    for label, dep in [("beta_Y", "ln_y"), ("beta_E", "ln_e"), ("beta_P", "ln_p")]:
        res = smf.ols(f"{dep} ~ ln_n + C({COL_COUNTRY})", data=d21).fit(cov_type=COV_TYPE)
        lo, hi = res.conf_int().loc["ln_n"]
        rows.append({"quantity": label, "slope": res.params["ln_n"],
                     "se_" + COV_TYPE: res.bse["ln_n"],
                     "ci95_low": lo, "ci95_high": hi, "n": int(res.nobs)})
    out_2021 = pd.DataFrame(rows)

    out_2021.to_csv(OUT_PATH_2021, index=False)
    print("Saved:", OUT_PATH_2021.as_posix())

print(out_2021.round(4).to_string(index=False))
s21 = out_2021.set_index("quantity")["slope"]
print(f"beta_Y - (beta_E + beta_P) = {s21['beta_Y'] - s21['beta_E'] - s21['beta_P']:.2e}")

In [ ]:
from pathlib import Path
import contextlib
import os
import sys
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.wkb
import skmob
import statsmodels.formula.api as smf
from geopandas.array import GeometryArray
from skmob.models.gravity import Gravity

# ---- config (change names and paths here only) ----
RES_DIR = Path("results/core_scaling_decomposition")
BAL_PATH = RES_DIR / "commuting_balance_2021.csv"
FULL_DELTA_PATH = RES_DIR / "null_model_delta_2021.csv"
MASTER_PATH = Path("data/processed/nuts3_master_table.parquet")
DIRS = {"CZ": Path("data/processed/05_gravity_cz"), "PL": Path("data/processed/05b_gravity_pl")}
MODELS_FIT = {"gravity_power": "power_law", "gravity_exp": "exponential"}
GRAVITY_TYPE = "singly constrained"
YEAR_GEO = 2023                      # same year as the tessellation of notebooks 05 / 05b
COL_ID, COL_COUNTRY, COL_YEAR = "nuts3_id", "country_code", "year"
COL_TILE, COL_POP, OUTFLOW_COL = "tile_ID", "population", "tot_outflows"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"
CRS_GEO = "EPSG:4326"
N_SPLITS = 5                         # number of random 2-fold splits of the origins
SEED_SPLIT = 0
SEED_BASE = 0
COV_TYPE = "HC3"
OUT_DELTA = RES_DIR / "null_model_delta_crossfit_2021.csv"
OUT_PARAMS = RES_DIR / "crossfit_params_2021.csv"
# ----------------------------------------------------

@contextlib.contextmanager
def quiet():
    """Silence C-level stderr (PROJ messages) and Python warnings inside the block."""
    sys.stderr.flush()
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 2)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)

def load_country(cc, folder):
    """Tessellation (geometry + population + outflows) and observed off-diagonal flows."""
    tess_csv = pd.read_csv(folder / "tessellation_outflows.csv")
    tess_csv[COL_TILE] = tess_csv[COL_TILE].astype(str)
    ids = tess_csv[COL_TILE].tolist()

    with quiet():
        # Geometry from the master table (stored as WKB bytes, assigned element-wise)
        geo = pd.read_parquet(MASTER_PATH, columns=[COL_ID, COL_COUNTRY, COL_YEAR, "geometry"])
        geo = geo[(geo[COL_COUNTRY] == cc) & (geo[COL_YEAR] == YEAR_GEO)].reset_index(drop=True)
        geoms = [shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x for x in geo["geometry"]]
        arr = np.empty(len(geoms), dtype=object)
        for i, g in enumerate(geoms):
            arr[i] = g
        geo = gpd.GeoDataFrame(geo.drop(columns="geometry"), geometry=GeometryArray(arr, crs=CRS_GEO))
        geo = geo.set_index(COL_ID).loc[ids].reset_index().rename(columns={COL_ID: COL_TILE})

        tess = geo[[COL_TILE, "geometry"]].merge(tess_csv[[COL_TILE, COL_POP, OUTFLOW_COL]], on=COL_TILE)
        assert tess[COL_TILE].tolist() == ids, f"{cc}: region order changed"
        tess = gpd.GeoDataFrame(tess, geometry="geometry", crs=CRS_GEO)

    # Observed flows: off-diagonal pairs with positive flow
    od = pd.read_csv(folder / "od_observed.csv", index_col=0)
    od.index, od.columns = od.index.astype(str), od.columns.astype(str)
    od = od.loc[ids, ids]
    flows = (od.rename_axis(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST)
               .stack().rename(COL_FLOW).reset_index())
    flows = flows[(flows[COL_FLOW_ORIGIN] != flows[COL_FLOW_DEST]) & (flows[COL_FLOW] > 0)]
    return {"ids": ids, "tess": tess, "flows": flows}

def fit_model(flows, tess, deterrence):
    """Fit one skmob Gravity model; keep only the warning class names."""
    with quiet():
        fdf = skmob.FlowDataFrame(flows, origin=COL_FLOW_ORIGIN, destination=COL_FLOW_DEST,
                                  flow=COL_FLOW, tessellation=tess, tile_id=COL_TILE)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
            model.fit(fdf, relevance_column=COL_POP)
    return model, ";".join(sorted({w.category.__name__ for w in caught}))

def expected_matrix(model, tess, ids):
    """Generate the expected flows of a fitted model as a matrix in the region order."""
    np.random.seed(SEED_BASE)
    with quiet():
        synth = model.generate(tess, tile_id_column=COL_TILE, tot_outflows_column=OUTFLOW_COL,
                               relevance_column=COL_POP, out_format="flows")
    return (pd.DataFrame(synth)
            .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
            .reindex(index=ids, columns=ids).fillna(0).to_numpy())

if OUT_DELTA.exists():
    delta = pd.read_csv(OUT_DELTA)
    params = pd.read_csv(OUT_PARAMS)
    print("Loaded existing:", OUT_DELTA.as_posix(), "and", OUT_PARAMS.as_posix())
else:
    data = {cc: load_country(cc, folder) for cc, folder in DIRS.items()}
    bal = pd.read_csv(BAL_PATH).set_index("nuts3_id")
    param_rows, delta_rows = [], []

    # Reference: fit on all origins (replication check against gravity_params.csv)
    for cc, d in data.items():
        for m, det in MODELS_FIT.items():
            model, w = fit_model(d["flows"], d["tess"], det)
            param_rows.append({"country": cc, "model": m, "split": "full", "fit_on": "all",
                               "destination_exp": model.destination_exp,
                               "gamma": model.deterrence_func_args[0], "fit_warnings": w})

    # Cross-fitting: fit on half of the origins, generate the rows of the other half
    for s in range(N_SPLITS):
        mats = {m: {} for m in MODELS_FIT}
        for cc, d in data.items():
            ids = d["ids"]
            perm = np.random.default_rng(SEED_SPLIT + s).permutation(len(ids))
            half = len(ids) // 2
            folds = {"A": {ids[i] for i in perm[:half]}, "B": {ids[i] for i in perm[half:]}}
            for m, det in MODELS_FIT.items():
                M = np.zeros((len(ids), len(ids)))
                for fit_fold, gen_fold in (("A", "B"), ("B", "A")):
                    fl = d["flows"][d["flows"][COL_FLOW_ORIGIN].isin(folds[fit_fold])]
                    model, w = fit_model(fl, d["tess"], det)
                    P = expected_matrix(model, d["tess"], ids)
                    rows = [i for i, r in enumerate(ids) if r in folds[gen_fold]]
                    M[rows, :] = P[rows, :]               # keep only rows unseen by the fit
                    param_rows.append({"country": cc, "model": m, "split": s, "fit_on": fit_fold,
                                       "destination_exp": model.destination_exp,
                                       "gamma": model.deterrence_func_args[0], "fit_warnings": w})
                mats[m][cc] = M

        # Delta_null of this split: E_hat = R + I_hat - X_hat, pooled CZ + PL with country FE
        for m in MODELS_FIT:
            parts = []
            for cc, d in data.items():
                M, sub = mats[m][cc], bal.loc[d["ids"]]
                parts.append(pd.DataFrame({
                    "country_code": cc, "R": sub["R_census"].to_numpy(),
                    "N": sub["population_2021"].to_numpy(),
                    "E_hat": sub["R_census"].to_numpy() + M.sum(axis=0) - M.sum(axis=1)}))
            g = pd.concat(parts, ignore_index=True)
            assert (g["E_hat"] > 0).all(), "non-positive E_hat found"
            dd = pd.DataFrame({"country_code": g["country_code"], "ln_n": np.log(g["N"]),
                               "dep": np.log(g["E_hat"] / g["R"])})
            res = smf.ols("dep ~ ln_n + C(country_code)", data=dd).fit(cov_type=COV_TYPE)
            delta_rows.append({"model": m, "split": s,
                               "delta_null": res.params["ln_n"], "se": res.bse["ln_n"]})

    delta, params = pd.DataFrame(delta_rows), pd.DataFrame(param_rows)
    delta.to_csv(OUT_DELTA, index=False)
    params.to_csv(OUT_PARAMS, index=False)
    print("Saved:", OUT_DELTA.as_posix(), "and", OUT_PARAMS.as_posix())

# Cross-fitted Delta_null next to the full-sample (non-independent) value
full = pd.read_csv(FULL_DELTA_PATH).groupby("model")["delta_null"].mean().rename("full_sample")
summ = delta.groupby("model")["delta_null"].agg(["mean", "std", "min", "max"]).join(full)
print(summ.round(4).to_string())

# Replication check: destination exponent of the full refit vs. notebooks 05 / 05b
ref = pd.concat([pd.read_csv(folder / "gravity_params.csv").assign(country=cc)
                 for cc, folder in DIRS.items()])[["country", "model", "destination_exp"]]
chk = (params[params["split"] == "full"][["country", "model", "destination_exp"]]
       .merge(ref, on=["country", "model"], suffixes=("_refit", "_notebook05")))
print(chk.round(4).to_string(index=False))

In [ ]:
# ---- Config (capital ids as in the robustness cell above) ----
CAPITALS = {                      # capital id -> label used in the output table
    "CZ010": "Praha",
    "HU110": "Budapest",
    "PL911": "Warszawa",
    "SK010": "Bratislava",
}
LOO_PATH = OUT_DIR / "core_scaling_leave_one_out.csv"
# --------------------------------------------------------------

# Reference fit: pooled, all regions, country fixed effects
reference = fit_three(df, with_country_fe=True)

rows = [{"sample": "pooled_all", "left_out": "none", **reference}]
for cap_id, cap_label in CAPITALS.items():
    # Leave out exactly one capital and refit
    subset = df[df[COL_ID] != cap_id]
    assert len(subset) == len(df) - 1, f"{cap_id} not found exactly once"
    rows.append({"sample": f"pooled_without_{cap_label}",
                 "left_out": cap_label,
                 **fit_three(subset, with_country_fe=True)})

loo = pd.DataFrame(rows)

# Change relative to the full pooled sample
for col in ["beta_Y", "beta_E", "beta_P"]:
    loo["delta_" + col] = loo[col] - reference[col]

show = ["left_out", "beta_Y", "beta_E", "beta_P", "n",
        "delta_beta_Y", "delta_beta_E", "delta_beta_P", "identity_diff"]
print(loo[show].round(4).to_string(index=False))

loo.to_csv(LOO_PATH, index=False)
print("Saved:", LOO_PATH.as_posix())